# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [8]:
# votre réponse ici
def lettre_vers_nombre(c):
    return ord(c) - ord("A")

def nombre_vers_lettre(n):
    return chr(n%26 + ord("A"))

print(lettre_vers_nombre('D'))
print(nombre_vers_lettre(25))

for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
    assert nombre_vers_lettre(lettre_vers_nombre(c)) == c

print("Pas d'erreur")

3
Z
Pas d'erreur


## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [15]:
# votre réponse ici
def nettoyer(texte):
    """Majuscules sans accents, lettres A-Z uniquement."""
    sans_accent = unicodedata.normalize("NFD", texte)
    sans_accent = "".join(c for c in sans_accent if unicodedata.category(c) != "Mn")
    return "".join(c for c in sans_accent.upper() if "A" <= c <= "Z")

print(nettoyer("Été, à Paris !"))

ETEAPARIS


## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [18]:
# votre réponse ici
def chiffrer(texte, a, b):
    resultat = ""
    for c in texte:
        resultat += nombre_vers_lettre((a*lettre_vers_nombre(c) + b) % 26)
    return resultat

chiffrer("ATTAQUEALAUBE", 5, 8)

'IZZIKECILIENC'

## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [34]:
# votre réponse ici
import math

def clefs_utilisables():
    utilisable = []
    for i in range (1, 26):
        if math.gcd(i, 26) == 1:
            utilisable.append(i)
    return utilisable

utilisable = clefs_utilisables()
print(utilisable, len(utilisable))

[1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25] 12


## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [24]:
# votre réponse ici
def inverse_modulaire(a, n):
    """Inverse de a modulo n, par recherche exhaustive."""
    a = a % n
    for candidat in range(1, n):
        if (a * candidat) % n == 1:        
            return candidat
    raise ValueError(f"{a} n'est pas inversible modulo {n}")

print(inverse_modulaire(5, 26))
print((5 * 21) % 26)

try:
    inverse_modulaire(4, 26)
except ValueError as erreur:    
    print("erreur attendue :", erreur)

21
1
erreur attendue : 4 n'est pas inversible modulo 26


## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [28]:
# votre réponse ici

def dechiffrer(texte, a, b):
    try:
        inv = inverse_modulaire(a, 26)
    except ValueError as error:
        print("a n'est pas premier avec 26 !")
        return ""
    dechiffree = ""
    for c in texte:
        dechiffree += nombre_vers_lettre(((lettre_vers_nombre(c)-b)*inv % 26))
    return dechiffree

print(dechiffrer(chiffrer(nettoyer("Test"), 5, 8), 5, 8))



TEST


## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [37]:
# votre réponse ici
def toutes_les_clefs(secret):
    possible = []
    for a in clefs_utilisables():
        for b in range(26):
            possible.append(dechiffrer(secret, a, b))
    return possible

secret = chiffrer(nettoyer("Test assez long à crypter et puis décrypter"), 5, 8)
a = toutes_les_clefs(secret)
print(a)
print(len(a))

['ZCUZIUUCDLAVMISPYFZCPCZFEWUXCSPYFZCP', 'YBTYHTTBCKZULHROXEYBOBYEDVTWBROXEYBO', 'XASXGSSABJYTKGQNWDXANAXDCUSVAQNWDXAN', 'WZRWFRRZAIXSJFPMVCWZMZWCBTRUZPMVCWZM', 'VYQVEQQYZHWRIEOLUBVYLYVBASQTYOLUBVYL', 'UXPUDPPXYGVQHDNKTAUXKXUAZRPSXNKTAUXK', 'TWOTCOOWXFUPGCMJSZTWJWTZYQORWMJSZTWJ', 'SVNSBNNVWETOFBLIRYSVIVSYXPNQVLIRYSVI', 'RUMRAMMUVDSNEAKHQXRUHURXWOMPUKHQXRUH', 'QTLQZLLTUCRMDZJGPWQTGTQWVNLOTJGPWQTG', 'PSKPYKKSTBQLCYIFOVPSFSPVUMKNSIFOVPSF', 'ORJOXJJRSAPKBXHENUOREROUTLJMRHENUORE', 'NQINWIIQRZOJAWGDMTNQDQNTSKILQGDMTNQD', 'MPHMVHHPQYNIZVFCLSMPCPMSRJHKPFCLSMPC', 'LOGLUGGOPXMHYUEBKRLOBOLRQIGJOEBKRLOB', 'KNFKTFFNOWLGXTDAJQKNANKQPHFINDAJQKNA', 'JMEJSEEMNVKFWSCZIPJMZMJPOGEHMCZIPJMZ', 'ILDIRDDLMUJEVRBYHOILYLIONFDGLBYHOILY', 'HKCHQCCKLTIDUQAXGNHKXKHNMECFKAXGNHKX', 'GJBGPBBJKSHCTPZWFMGJWJGMLDBEJZWFMGJW', 'FIAFOAAIJRGBSOYVELFIVIFLKCADIYVELFIV', 'EHZENZZHIQFARNXUDKEHUHEKJBZCHXUDKEHU', 'DGYDMYYGHPEZQMWTCJDGTGDJIAYBGWTCJDGT', 'CFXCLXXFGODYPLVSBICFSFCIHZXAFVSBICFS', 'BEWBKWWEFNCXOKURAHBEREBHGYWZEURAHBER',

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [39]:
# votre réponse ici
def force_brute_auto(secret):
    possibles = toutes_les_clefs(secret)
    for possibilite in possibles:
        occurences = {}
        for c in possibilite:
            if c not in occurences:
                occurences[c] = 1
            else:
                occurences[c] += 1
        if 'E' in occurences:
            freq = occurences['E']/len(possibilite)
            if freq == max([a/len(possibilite) for a in occurences.values()]):
                return possibilite
    return ""

secret = chiffrer(nettoyer("Test assez long à crypter et puis décrypter"), 5, 8)
print(force_brute_auto(secret))

secret = chiffrer(nettoyer("Ceci est un long message avec beaucoup de mot tel que la fréquence des lettres est proche de celle de la langue française en général"), 5, 8)
print(force_brute_auto(secret))

'UEUYEWBGXNCXOSEWWKOEKLEUPEKGUCGHZESCBBENMGENKJREMGEXUEZEWNEBBREWEWBHRCUTEZEUENNEZENKNKXOGEJRKXUKYWEEXOEXERKN'

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [ ]:
# votre réponse ici


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [ ]:
# votre réponse ici


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [ ]:
# votre réponse ici
